# Smart Power Switch Daily Forecast Model

This notebook reads the live workbook structure from the system, trains an LSTM model on the `Daily Data` sheet, and builds a 30-day forecast for consumption and bill estimation.

Inputs used from the workbook:
- `Cluster` as the daily date label
- `Energy (kWh)` as the target series
- `Cost (PHP)` as the bill series

In [2]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

import warnings, os
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
np.random.seed(42)

ModuleNotFoundError: No module named 'pandas'

## Load the workbook structure

The source workbook is already organized, so we read the `Daily Data` sheet directly instead of converting a raw TXT file.

In [1]:
WORKBOOK_FILE = r'c:/Users/Rouransu/Downloads/c36-11f1-90bd-f72804fc8fe6.xlsx'
OUTPUT_IMG = 'smartpowerswitch_daily_prediction_comparison.png'
TEST_DAYS = 30
LOOK_BACK = 14
RANDOM_SEED = 42

df = pd.read_excel(WORKBOOK_FILE, sheet_name='Daily Data', skiprows=9)
df = df.rename(columns={
    'Cluster': 'Date',
    'Energy (kWh)': 'Daily_kWh',
    'Cost (PHP)': 'Daily_Cost'
})
df = df[['Date', 'Daily_kWh', 'Daily_Cost']].copy()
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['Daily_kWh'] = pd.to_numeric(df['Daily_kWh'], errors='coerce')
df['Daily_Cost'] = pd.to_numeric(df['Daily_Cost'], errors='coerce')
df = df.dropna(subset=['Date', 'Daily_kWh', 'Daily_Cost']).reset_index(drop=True)
df['Day_Num'] = range(1, len(df) + 1)

print('Rows loaded:', len(df))
print('Date range:', df['Date'].min().date(), '→', df['Date'].max().date())
print(df.head().to_string(index=False))

NameError: name 'pd' is not defined

## Prepare train/test split

We use the daily kWh series as the main model target. The bill is estimated from the average rate observed in the historical data.

In [ ]:
dates_all = [str(d.date()) for d in df['Date']]
y = df['Daily_kWh'].values.astype(float)
cost_y = df['Daily_Cost'].values.astype(float)
X = df['Day_Num'].values.reshape(-1, 1)

split = len(df) - TEST_DAYS
if split <= LOOK_BACK + 5:
    raise ValueError('Not enough daily rows for the selected test window and look-back window.')

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
dates_train, dates_test = dates_all[:split], dates_all[split:]

rate_series = np.divide(cost_y, np.where(y == 0, np.nan, y))
avg_rate = np.nanmean(rate_series)
print('Average rate from data:', round(float(avg_rate), 4), 'PHP/kWh')

In [ ]:
def evaluate(y_true, y_pred, label):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, np.nan, y_true))) * 100
    print(f'{label:18} MAE={mae:.3f} RMSE={rmse:.3f} R2={r2:.4f} MAPE={mape:.2f}%')
    return mae, rmse, r2, mape

## LSTM model

We train the sequence model on the normalized daily kWh series from the workbook.

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_train_pred = lr_model.predict(X_train)
lr_test_pred = lr_model.predict(X_test)
lr_future = lr_model.predict(np.array([[len(df) + 1]]))[0]
lr_mae, lr_rmse, lr_r2, lr_mape = evaluate(y_test, lr_test_pred, 'Linear Regression')

dt_model = DecisionTreeRegressor(random_state=RANDOM_SEED, max_depth=6)
dt_model.fit(X_train, y_train)
dt_train_pred = dt_model.predict(X_train)
dt_test_pred = dt_model.predict(X_test)
dt_future = dt_model.predict(np.array([[len(df) + 1]]))[0]
dt_mae, dt_rmse, dt_r2, dt_mape = evaluate(y_test, dt_test_pred, 'Decision Tree')

## LSTM model

We train the sequence model on a normalized daily kWh series.

In [ ]:
scaler = MinMaxScaler()
scaled = scaler.fit_transform(y.reshape(-1, 1)).flatten()

def make_sequences(series, look_back):
    xs, ys = [], []
    for i in range(len(series) - look_back):
        xs.append(series[i:i + look_back])
        ys.append(series[i + look_back])
    return np.array(xs), np.array(ys)

X_seq, y_seq = make_sequences(scaled, LOOK_BACK)
seq_split = split - LOOK_BACK
X_seq_train, X_seq_test = X_seq[:seq_split], X_seq[seq_split:]
y_seq_train, y_seq_test = y_seq[:seq_split], y_seq[seq_split:]

X_seq_train = X_seq_train.reshape((X_seq_train.shape[0], X_seq_train.shape[1], 1))
X_seq_test = X_seq_test.reshape((X_seq_test.shape[0], X_seq_test.shape[1], 1))

lstm_model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(LOOK_BACK, 1)),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])
lstm_model.compile(optimizer='adam', loss='mse')
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = lstm_model.fit(
    X_seq_train, y_seq_train,
    validation_split=0.1,
    epochs=60,
    batch_size=16,
    callbacks=[early_stop],
    verbose=1
)

lstm_train_pred_scaled = lstm_model.predict(X_seq_train, verbose=0).flatten()
lstm_test_pred_scaled = lstm_model.predict(X_seq_test, verbose=0).flatten()
lstm_train_pred = scaler.inverse_transform(lstm_train_pred_scaled.reshape(-1, 1)).flatten()
lstm_test_pred = scaler.inverse_transform(lstm_test_pred_scaled.reshape(-1, 1)).flatten()
lstm_future_seq = scaled[-LOOK_BACK:].reshape(1, LOOK_BACK, 1)
lstm_future = scaler.inverse_transform(lstm_model.predict(lstm_future_seq, verbose=0))[0][0]
lstm_mae = mean_absolute_error(y[LOOK_BACK + seq_split:], lstm_test_pred)
lstm_rmse = np.sqrt(mean_squared_error(y[LOOK_BACK + seq_split:], lstm_test_pred))
lstm_r2 = r2_score(y[LOOK_BACK + seq_split:], lstm_test_pred)
lstm_mape = np.mean(np.abs((y[LOOK_BACK + seq_split:] - lstm_test_pred) / np.where(y[LOOK_BACK + seq_split:] == 0, np.nan, y[LOOK_BACK + seq_split:]))) * 100
print(f'LSTM              MAE={lstm_mae:.3f} RMSE={lstm_rmse:.3f} R2={lstm_r2:.4f} MAPE={lstm_mape:.2f}%')

## Forecast next month

We extend the daily kWh forecast for 30 days and estimate the bill using the historical average rate.

In [ ]:
def forecast_next_30_days():
    future_dates = []
    future_kwh = []

    last_date = df['Date'].iloc[-1]
    for i in range(1, 31):
        future_dates.append((last_date + pd.Timedelta(days=i)).date())

    seq = scaled[-LOOK_BACK:].copy()
    for _ in range(30):
        pred_scaled = lstm_model.predict(seq.reshape(1, LOOK_BACK, 1), verbose=0)[0, 0]
        pred = float(scaler.inverse_transform([[pred_scaled]])[0, 0])
        future_kwh.append(max(pred, 0.0))
        seq = np.append(seq[1:], pred_scaled)

    future_bill = [value * avg_rate for value in future_kwh]
    return future_dates, future_kwh, future_bill

future_dates, future_kwh, future_bill = forecast_next_30_days()
print('Next 30-day projected kWh total:', round(sum(future_kwh), 2))
print('Next 30-day projected bill:', round(sum(future_bill), 2))

## Visualization

The notebook shows the LSTM results and places the forecast graph below the main history chart.

In [ ]:
COLORS = {
    'lstm': '#DC2626',
    'actual': '#1E293B',
    'forecast': '#F59E0B',
    'bg': '#F8FAFC',
}

HIST_WINDOW = 90
dates_hist = dates_all[-HIST_WINDOW:]
y_hist = y[-HIST_WINDOW:]

fig = plt.figure(figsize=(22, 22), facecolor=COLORS['bg'])
fig.suptitle(
    'Smart Power Switch Daily Consumption Prediction\nLSTM Forecast',
    fontsize=18, fontweight='bold', color=COLORS['actual'], y=0.98
)
gs = gridspec.GridSpec(3, 1, figure=fig, hspace=0.45, top=0.94, bottom=0.04, left=0.07, right=0.97)

ax0 = fig.add_subplot(gs[0, 0])
ax0.set_facecolor('white')
ax0.fill_between(range(len(y_hist)), y_hist, alpha=0.25, color='#94A3B8')
ax0.plot(range(len(y_hist)), y_hist, '-', color=COLORS['actual'], lw=1.5, label='Daily kWh')
ax0.set_xticks(range(0, len(dates_hist), 7))
ax0.set_xticklabels([dates_hist[i] for i in range(0, len(dates_hist), 7)], rotation=45, ha='right', fontsize=8)
ax0.set_title(f'Historical Daily Electricity Consumption — Last {HIST_WINDOW} Days (kWh)', fontweight='bold', fontsize=13, pad=10)
ax0.set_ylabel('kWh')
ax0.legend(fontsize=10)
ax0.grid(axis='y', alpha=0.3)
ax0.spines[['top', 'right']].set_visible(False)

ax1 = fig.add_subplot(gs[1, 0])
ax1.set_facecolor('white')
ax1.plot(dates_all[LOOK_BACK:split][-60:], y[LOOK_BACK:split][-60:], '-', color='#94A3B8', lw=1.2, label='Train Actual')
ax1.plot(dates_test, y[split:], 's-', color=COLORS['actual'], lw=2, ms=5, label='Test Actual')
ax1.plot(dates_all[LOOK_BACK:split][-60:], lstm_train_pred[-60:], '--', color=COLORS['lstm'], lw=1.5, alpha=0.7, label='Train Pred')
ax1.plot(dates_test, lstm_test_pred, 's--', color=COLORS['lstm'], lw=2, ms=5, label='Test Pred')
ax1.axvline(x=dates_test[0], color='gray', ls=':', lw=1.5, alpha=0.6)
ax1.set_title('LSTM (Long Short-Term Memory) — 14-Day Look-Back Window', fontweight='bold', fontsize=12, color=COLORS['lstm'])
ax1.set_ylabel('kWh')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)
ax1.tick_params(axis='x', rotation=35, labelsize=8)
all_dates_lstm = dates_all[LOOK_BACK:split][-60:] + dates_test
ax1.set_xticks([all_dates_lstm[i] for i in range(0, len(all_dates_lstm), 7)])
ax1.spines[['top', 'right']].set_visible(False)
ax1.text(0.99, 0.95, f'MAE: {lstm_mae:.2f}  RMSE: {lstm_rmse:.2f}  R²: {lstm_r2:.4f}  MAPE: {lstm_mape:.2f}%', transform=ax1.transAxes, ha='right', va='top', fontsize=9, bbox=dict(boxstyle='round', fc='#FFF1F2', ec=COLORS['lstm'], alpha=0.9))

ax2 = fig.add_subplot(gs[2, 0])
ax2.set_facecolor('white')
ax2.plot(dates_all[-60:], y[-60:], '-', color='#94A3B8', lw=1.4, label='Actual (last 60 days)')
ax2.plot(future_dates, np.array(future_kwh), 'o--', color=COLORS['forecast'], lw=2, ms=4, label='Forecast next 30 days')
ax2.axvline(x=dates_all[-1], color='gray', ls=':', lw=1.5, alpha=0.7)
ax2.set_title('Prediction Graph Below Main Charts', fontweight='bold', fontsize=12, color=COLORS['forecast'])
ax2.set_ylabel('kWh')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)
ax2.tick_params(axis='x', rotation=35, labelsize=8)
ax2.spines[['top', 'right']].set_visible(False)
ax2.text(0.5, -0.28, f'Forecast next 30 days\nTotal kWh: {sum(future_kwh):.2f}\nEstimated bill: ₱ {sum(future_bill):.2f}', transform=ax2.transAxes, ha='center', va='top', fontsize=10, family='monospace', bbox=dict(boxstyle='round', fc='#FEF9C3', ec='#CA8A04', alpha=0.95))

plt.savefig(OUTPUT_IMG, dpi=150, bbox_inches='tight', facecolor=COLORS['bg'])
print('Chart saved ->', OUTPUT_IMG)